# Function 4: 4D black-box maximisation

Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

## How this notebook works

1. **Data loading:** the incumbent (best observed `y`) is derived from `data.csv`.
2. **Primary GP:** a constant × Matérn 3/2 ARD Gaussian process on standardised inputs (kernel, target and alpha chosen earlier by leave-one-out CV), validated with leave-one-out CV.
3. **Local GP acquisition:** a Sobol sample of a box around the incumbent, excluding points already evaluated, scored by posterior mean, EI, PI and UCB.
4. **Neural-network challenger:** an ensemble of small MLPs, validated by leave-one-out CV and used to rank random candidates around the incumbent.
5. **TuRBO challenger:** an independent Matérn 5/2 GP with a trust region centred on the incumbent and a joint Thompson sample.
6. **Comparison and recommendation:** every candidate is scored by all three models in one table. The GP posterior-mean point is printed in submission format.

Everything is derived from `data.csv`: append new rows and re-run, with no code edits.

**Data-handling choices.**

- *Rare positives.* Few observations are positive, so exploration away from the confirmed positive basin is costly; candidates are searched locally and previously evaluated points are excluded.
- *Model independence.* The GP, neural-network and TuRBO models are trained on the same observations and can share errors, so agreement between them is a check, not independent evidence.
- *Input format.* Candidates are printed to 6 decimals, as the portal requires.

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import warnings

import numpy as np
import pandas as pd

from scipy.stats import norm, qmc
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# GP fitting regularly reports optimiser convergence warnings; they do not
# change the fitted models, so they are suppressed.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================
# Loads the centrally prepared BBO dataset and selects the active function.
# Active input dimensions are inferred from the x-columns that contain data.

DATA_PATH = "../data/data.csv"
FUNCTION_ID = 4

data = pd.read_csv(DATA_PATH, float_precision="round_trip")
function_data = data[data["function"] == FUNCTION_ID].copy()

feature_cols = [
    f"x{i}" for i in range(1, 9)
    if function_data[f"x{i}"].notna().any()
]

# Modelling data (feature DataFrame and target Series, with the original index)
df = function_data[[*feature_cols, "y", "round"]].copy()
df["Stage"] = df["round"].apply(
    lambda r: "Initial Data" if r == 0 else f"Week {r}"
)
df = df.rename(columns={"round": "Round"})
df = df[[*feature_cols, "y", "Stage", "Round"]]

X = df[feature_cols]
y = df["y"]
n_features = len(feature_cols)

# Dynamic incumbent: highest OBSERVED objective value
incumbent = df.loc[y.idxmax()]
incumbent_x = incumbent[feature_cols].to_numpy(dtype=float)
incumbent_y = float(incumbent["y"])
incumbent_round = incumbent["Round"]

# All previously evaluated configurations (used to avoid duplicate proposals)
evaluated_X = X.to_numpy(dtype=float)

print(f"Function {FUNCTION_ID} observations: {len(df)} | Active features: {feature_cols}")
print(f"Incumbent: R{incumbent_round}, y = {incumbent_y:.9f}, x = {np.round(incumbent_x, 6)}")


def format_point(x):
    """Format a point as the dash-separated six-decimal submission string."""
    return "-".join(f"{v:.6f}" for v in x)

Function 4 observations: 42 | Active features: ['x1', 'x2', 'x3', 'x4']
Incumbent: R8, y = 0.730826293, x = [0.42 0.36 0.36 0.42]


In [3]:
# =============================================================================
# PRIMARY MODEL: GAUSSIAN PROCESS
# =============================================================================
# Function 4 is a MAXIMISATION problem. The surrogate is an ARD Matérn-3/2 GP on
# standardised inputs with a raw (untransformed) target and alpha = 0.01. This
# specification was selected by leave-one-out CV over kernel family
# (Matérn-3/2, Matérn-5/2, RBF), target transform (raw, Yeo-Johnson) and
# alpha (1e-4 ... 1e-1): it had the lowest LOOCV MAE (1.80, R² = 0.92).

GP_ALPHA = 0.01
GP_CV_N_RESTARTS = 5
GP_FINAL_N_RESTARTS = 25
GP_RANDOM_STATE = 42

GP_KERNEL = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
    length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e3), nu=1.5
)


def build_gp_pipeline(n_restarts):
    """Standardise inputs, then fit the GP."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianProcessRegressor(
            kernel=clone(GP_KERNEL), alpha=GP_ALPHA,
            n_restarts_optimizer=n_restarts, random_state=GP_RANDOM_STATE)),
    ])

def predict_gp_mean_std(pipeline, X_new):
    """GP posterior mean and standard deviation (original y units) for points X_new."""
    X_input = pd.DataFrame(np.atleast_2d(np.asarray(X_new, dtype=float)), columns=feature_cols)
    X_scaled = pipeline.named_steps["scaler"].transform(X_input)
    mean, std = pipeline.named_steps["model"].predict(X_scaled, return_std=True)
    return np.asarray(mean).ravel(), np.asarray(std).ravel()

def distance_to_evaluated(X_candidates):
    """Distance from each candidate to its nearest previously evaluated configuration."""
    candidates = np.atleast_2d(np.asarray(X_candidates, dtype=float))
    return np.linalg.norm(candidates[:, None, :] - evaluated_X[None, :, :], axis=2).min(axis=1)

# Leave-one-out validation of the chosen specification
y_pred_cv = cross_val_predict(build_gp_pipeline(GP_CV_N_RESTARTS), X, y, cv=LeaveOneOut(), n_jobs=-1)
print(f"GP LOOCV: MAE={mean_absolute_error(y, y_pred_cv):.6f}, "
      f"RMSE={np.sqrt(np.mean((y.to_numpy() - y_pred_cv) ** 2)):.6f}, R²={r2_score(y, y_pred_cv):.6f}")

# Final GP on all observations
best_pipeline = build_gp_pipeline(GP_FINAL_N_RESTARTS).fit(X, y)
fitted_kernel = best_pipeline.named_steps["model"].kernel_
length_scales = np.atleast_1d(fitted_kernel.k2.length_scale)
print(f"Fitted kernel: {fitted_kernel}")
print("ARD sensitivity (1 / length scale): "
      + ", ".join(f"{f}={1 / ls:.3f}" for f, ls in zip(feature_cols, length_scales)))

incumbent_gp_mean, incumbent_gp_std = predict_gp_mean_std(best_pipeline, incumbent_x)
print(f"Incumbent: observed y = {incumbent_y:.6f} | GP mean = {incumbent_gp_mean[0]:.6f} "
      f"| GP std = {incumbent_gp_std[0]:.6f}")

GP LOOCV: MAE=1.741741, RMSE=2.696002, R²=0.924599
Fitted kernel: 10**2 * Matern(length_scale=[4.35, 3.28, 4.38, 3.99], nu=1.5)
ARD sensitivity (1 / length scale): x1=0.230, x2=0.305, x3=0.228, x4=0.250
Incumbent: observed y = 0.730826 | GP mean = 0.640893 | GP std = 0.063722


In [4]:
# =============================================================================
# LOCAL GP ACQUISITION CANDIDATES
# =============================================================================
# Scores a Sobol sample of the box around the incumbent (previously evaluated
# points excluded) with the GP: posterior mean, EI, PI and UCB.

GP_TRUST_RADIUS = 0.05       # half-width of the search box around the incumbent
GP_LOCAL_SAMPLES = 16384
GP_MIN_EVAL_DISTANCE = 0.001
GP_UCB_BETA = 2.0
GP_ACQUISITION_SEED = 42

bounds_lo = np.maximum(0.0, incumbent_x - GP_TRUST_RADIUS)
bounds_hi = np.minimum(1.0, incumbent_x + GP_TRUST_RADIUS)
sampler = qmc.Sobol(d=len(feature_cols), scramble=True, seed=GP_ACQUISITION_SEED)
search_X = qmc.scale(sampler.random_base2(int(np.log2(GP_LOCAL_SAMPLES))), bounds_lo, bounds_hi)
search_X = search_X[distance_to_evaluated(search_X) >= GP_MIN_EVAL_DISTANCE]

mu, sigma = predict_gp_mean_std(best_pipeline, search_X)
sigma = np.maximum(sigma, 1e-12)
z = (mu - incumbent_y) / sigma
scores = {
    "GP_Posterior_Mean": mu,
    "GP_EI": (mu - incumbent_y) * norm.cdf(z) + sigma * norm.pdf(z),
    "GP_PI": norm.cdf(z),
    "GP_UCB": mu + GP_UCB_BETA * sigma,
}

rows = []
for method, score in scores.items():
    idx = int(np.argmax(score))
    rows.append({
        "Method": method,
        **dict(zip(feature_cols, search_X[idx])),
        "GP_pred_y": float(mu[idx]),
        "GP_pred_std": float(sigma[idx]),
        "Acquisition_score": float(score[idx]),
        "Distance_to_Evaluated": float(distance_to_evaluated(search_X[idx:idx + 1])[0]),
    })
local_acquisition_results = pd.DataFrame(rows)
gp_pm_x = local_acquisition_results.loc[
    local_acquisition_results["Method"] == "GP_Posterior_Mean", feature_cols
].iloc[0].to_numpy(dtype=float)

print("LOCAL GP ACQUISITION CANDIDATES")
print(local_acquisition_results.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\nRecommended next query (GP posterior mean):")
print(format_point(gp_pm_x))

LOCAL GP ACQUISITION CANDIDATES
           Method       x1       x2       x3       x4  GP_pred_y  GP_pred_std  Acquisition_score  Distance_to_Evaluated
GP_Posterior_Mean 0.422624 0.350863 0.359345 0.422000   0.644108     0.095780           0.644108               0.009737
            GP_EI 0.449505 0.333625 0.367610 0.455359   0.296419     0.475891           0.046681               0.053613
            GP_PI 0.426753 0.348519 0.359917 0.433182   0.602153     0.185240           0.243644               0.018740
           GP_UCB 0.456118 0.331285 0.367373 0.469275   0.090345     0.601214           1.292772               0.067907

Recommended next query (GP posterior mean):
0.422624-0.350863-0.359345-0.422000


In [5]:
# =============================================================================
# CHALLENGER 1: SMALL NEURAL-NETWORK ENSEMBLE
# =============================================================================
# An independent local ranking challenger. An ensemble of small MLPs (different
# seeds) is fitted to all data and scored on random points in the box around
# the incumbent. The ensemble spread measures model instability, not Bayesian
# uncertainty.

NN_HIDDEN_LAYERS = (16, 8)
NN_ALPHA = 0.01
NN_MAX_ITER = 3000
NN_LOO_SEEDS = 10             # seeds averaged in the leave-one-out check
NN_SEARCH_ENSEMBLE = 50       # networks in the final ensemble
NN_TRUST_RADIUS = 0.05
NN_RANDOM_CANDIDATES = 200_000
NN_BASE_RANDOM_STATE = 100
NN_MIN_EVAL_DISTANCE = 0.02


def build_nn(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(hidden_layer_sizes=NN_HIDDEN_LAYERS, activation="relu",
                               solver="adam", alpha=NN_ALPHA, max_iter=NN_MAX_ITER,
                               random_state=seed)),
    ])

y_nn = y.to_numpy(dtype=float)

# Leave-one-out validation, averaged over initialisations
nn_loo_pred = np.vstack([
    cross_val_predict(build_nn(NN_BASE_RANDOM_STATE + seed), X, y_nn, cv=LeaveOneOut(), n_jobs=-1)
    for seed in range(NN_LOO_SEEDS)
]).mean(axis=0)
print(f"NN LOOCV: MAE={mean_absolute_error(y_nn, nn_loo_pred):.6f}, "
      f"RMSE={np.sqrt(np.mean((y_nn - nn_loo_pred) ** 2)):.6f}, R²={r2_score(y_nn, nn_loo_pred):.6f}")

# Random candidates around the incumbent, excluding evaluated / near-duplicate points
rng = np.random.default_rng(NN_BASE_RANDOM_STATE)
X_nn_search = rng.uniform(np.maximum(incumbent_x - NN_TRUST_RADIUS, 0.0),
                          np.minimum(incumbent_x + NN_TRUST_RADIUS, 1.0),
                          size=(NN_RANDOM_CANDIDATES, n_features))
nn_distances = distance_to_evaluated(X_nn_search)
valid = nn_distances >= NN_MIN_EVAL_DISTANCE
X_nn_search, nn_distances = X_nn_search[valid], nn_distances[valid]
X_nn_search_df = pd.DataFrame(X_nn_search, columns=feature_cols)

# Fit the ensemble and score every candidate
print(f"Fitting {NN_SEARCH_ENSEMBLE} NNs; evaluating {len(X_nn_search):,} candidates...")
nn_ensemble_models = [build_nn(NN_BASE_RANDOM_STATE + seed).fit(X, y_nn)
                      for seed in range(NN_SEARCH_ENSEMBLE)]
nn_preds = np.vstack([m.predict(X_nn_search_df) for m in nn_ensemble_models])
nn_mean = nn_preds.mean(axis=0)
nn_spread = np.sqrt(np.maximum((nn_preds ** 2).mean(axis=0) - nn_mean ** 2, 0.0))

nn_best_idx = int(np.argmax(nn_mean))
nn_candidate = X_nn_search[nn_best_idx].copy()

print("\nNN CHALLENGER")
print(format_point(nn_candidate))
print(f"Ensemble mean: {nn_mean[nn_best_idx]:.6f} | spread: {nn_spread[nn_best_idx]:.6f}")
print(f"Distance to incumbent: {np.linalg.norm(nn_candidate - incumbent_x):.6f} "
      f"| to evaluated: {nn_distances[nn_best_idx]:.6f}")

NN LOOCV: MAE=1.334994, RMSE=1.810194, R²=0.966007
Fitting 50 NNs; evaluating 191,183 candidates...

NN CHALLENGER
0.436031-0.396162-0.385686-0.438216
Ensemble mean: 0.402572 | spread: 0.288303
Distance to incumbent: 0.050560 | to evaluated: 0.020811


In [6]:
# =============================================================================
# CHALLENGER 2: TuRBO-1
# =============================================================================
# An independent trust-region Bayesian optimisation challenger with its own
# near-deterministic Matérn-5/2 GP. The box is centred on the incumbent with a
# fixed side length (the historical queries were not TuRBO-directed, so the box
# is not adapted from them) and shaped by the GP length scales. A joint
# Thompson sample over Sobol candidates in the box picks the point.

TURBO_SEED = 15
TURBO_LENGTH = 0.15            # box side length (fraction of the unit cube)
TURBO_N_CANDIDATES = 1024
TURBO_N_RESTARTS = 15
TURBO_GP_ALPHA = 1e-6
TURBO_MIN_EVAL_DISTANCE = 0.001

X_turbo, y_turbo = X.to_numpy(dtype=float), y.to_numpy(dtype=float)
d_turbo = X_turbo.shape[1]
turbo_y_mean, turbo_y_std = y_turbo.mean(), y_turbo.std()

turbo_gp = GaussianProcessRegressor(
    kernel=ConstantKernel(1.0, (1e-2, 1e2))
    * Matern(length_scale=np.full(d_turbo, 0.3), length_scale_bounds=(0.01, 10.0), nu=2.5),
    alpha=TURBO_GP_ALPHA, n_restarts_optimizer=TURBO_N_RESTARTS,
    random_state=TURBO_SEED, normalize_y=False,
).fit(X_turbo, (y_turbo - turbo_y_mean) / turbo_y_std)

# Length-scale-shaped trust region centred on the incumbent
turbo_centre = incumbent_x.copy()
turbo_length_scales = np.asarray(turbo_gp.kernel_.k2.length_scale, dtype=float)
turbo_weights = turbo_length_scales / np.exp(np.mean(np.log(turbo_length_scales)))
turbo_lower = np.clip(turbo_centre - turbo_weights * TURBO_LENGTH / 2, 0, 1)
turbo_upper = np.clip(turbo_centre + turbo_weights * TURBO_LENGTH / 2, 0, 1)

# Sobol candidates, each perturbing a random subset of coordinates
turbo_rng = np.random.default_rng(TURBO_SEED)
turbo_sobol_points = qmc.Sobol(d_turbo, scramble=True, seed=TURBO_SEED).random_base2(
    int(np.ceil(np.log2(TURBO_N_CANDIDATES))))[:TURBO_N_CANDIDATES]
turbo_perturbations = turbo_lower + (turbo_upper - turbo_lower) * turbo_sobol_points

turbo_mask = turbo_rng.random(turbo_perturbations.shape) <= min(1.0, 20.0 / d_turbo)
turbo_mask[np.arange(len(turbo_mask)), turbo_rng.integers(d_turbo, size=len(turbo_mask))] = True
turbo_candidates = np.where(turbo_mask, turbo_perturbations, turbo_centre)

# Drop previously evaluated and near-duplicate configurations
turbo_distances = distance_to_evaluated(turbo_candidates)
turbo_valid = turbo_distances >= TURBO_MIN_EVAL_DISTANCE
turbo_candidates, turbo_distances = turbo_candidates[turbo_valid], turbo_distances[turbo_valid]

# Joint Thompson sample (preserves correlations between candidates)
turbo_draw = turbo_gp.sample_y(turbo_candidates, n_samples=1, random_state=TURBO_SEED).ravel()
turbo_best_idx = int(np.argmax(turbo_draw))
turbo_candidate = turbo_candidates[turbo_best_idx].copy()

turbo_mu, turbo_sigma = turbo_gp.predict(turbo_candidate.reshape(1, -1), return_std=True)

print(f"Fitted kernel: {turbo_gp.kernel_}")
print("Trust region: " + ", ".join(f"{f} [{lo:.4f}, {hi:.4f}]"
                                   for f, lo, hi in zip(feature_cols, turbo_lower, turbo_upper)))
print("\nTuRBO CHALLENGER")
print(format_point(turbo_candidate))
print(f"Predicted y: {turbo_mu[0] * turbo_y_std + turbo_y_mean:.6f} "
      f"| GP predictive SD: {turbo_sigma[0] * turbo_y_std:.6f}")
print(f"Distance to incumbent: {np.linalg.norm(turbo_candidate - turbo_centre):.6f} "
      f"| to evaluated: {turbo_distances[turbo_best_idx]:.6f}")

Fitted kernel: 0.926**2 * Matern(length_scale=[0.63, 0.179, 0.76, 0.68], nu=2.5)
Trust region: x1 [0.3238, 0.5162], x2 [0.3327, 0.3873], x3 [0.2439, 0.4761], x4 [0.3161, 0.5239]

TuRBO CHALLENGER
0.502548-0.344536-0.345694-0.421634
Predicted y: 1.096725 | GP predictive SD: 0.681114
Distance to incumbent: 0.085210 | to evaluated: 0.083993


In [7]:
# =============================================================================
# CANDIDATE COMPARISON: GP + NN + TuRBO
# =============================================================================
# Every candidate is scored by all three models.
# Candidates that coincide (e.g. GP posterior mean and GP PI) are listed once.
# GP    : primary surrogate (posterior mean and approximate std).
# NN    : ensemble mean and spread (instability, not calibrated uncertainty).
# TuRBO : independent trust-region GP (mean and std).

# The code selects the best candidate in four steps:
# 1. Starts with the best observed point: Round 8, which achieved 0.730826.
# 2. Generates 16,384 candidate points in the neighbourhood of Round 8.
# 3. Uses the GP model to predict y for each candidate.
# 4. Selects the candidate with the highest predicted y.

candidate_lookup = {
    f"R{incumbent_round}_Incumbent": incumbent_x.copy(),
    "NN_Challenger": nn_candidate.copy(),
    "TuRBO_Challenger": turbo_candidate.copy(),
}
for _, row in local_acquisition_results.iterrows():
    candidate_lookup[row["Method"]] = row[feature_cols].to_numpy(dtype=float)

unique_candidates = {}
for name, point in candidate_lookup.items():
    if not any(np.allclose(point, previous, rtol=0, atol=1e-10) for previous in unique_candidates.values()):
        unique_candidates[name] = point
candidate_X = pd.DataFrame(list(unique_candidates.values()), index=list(unique_candidates), columns=feature_cols)

gp_mean, gp_std = predict_gp_mean_std(best_pipeline, candidate_X)
nn_predictions = np.vstack([model.predict(candidate_X) for model in nn_ensemble_models])
turbo_mean_scaled, turbo_std_scaled = turbo_gp.predict(candidate_X.to_numpy(), return_std=True)

candidate_comparison = candidate_X.reset_index(names="Candidate")
candidate_comparison["Distance_to_Incumbent"] = np.linalg.norm(candidate_X.to_numpy() - incumbent_x, axis=1)
candidate_comparison["Distance_to_Evaluated"] = distance_to_evaluated(candidate_X.to_numpy())
candidate_comparison["GP_mean"], candidate_comparison["GP_std"] = gp_mean, gp_std
candidate_comparison["NN_mean"], candidate_comparison["NN_spread"] = nn_predictions.mean(axis=0), nn_predictions.std(axis=0)
candidate_comparison["TuRBO_mean"] = turbo_mean_scaled * turbo_y_std + turbo_y_mean
candidate_comparison["TuRBO_std"] = turbo_std_scaled * turbo_y_std

print(f"Observations: {len(df)} | Incumbent: R{incumbent_round} | Best y: {incumbent_y:.9f}\n")
print(candidate_comparison.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("\nRecommended next query (GP posterior mean):")
print(format_point(gp_pm_x))

Observations: 42 | Incumbent: R8 | Best y: 0.730826293

        Candidate       x1       x2       x3       x4  Distance_to_Incumbent  Distance_to_Evaluated   GP_mean   GP_std   NN_mean  NN_spread  TuRBO_mean  TuRBO_std
     R8_Incumbent 0.420000 0.360000 0.360000 0.420000               0.000000               0.000000  0.640893 0.063722  0.333035   0.247089    0.723962   0.009189
    NN_Challenger 0.436031 0.396162 0.385686 0.438216               0.050560               0.020811  0.342186 0.124338  0.402572   0.288303    0.180546   0.104963
 TuRBO_Challenger 0.502548 0.344536 0.345694 0.421634               0.085210               0.083993 -0.237479 0.596505 -0.400163   0.477751    1.096725   0.681114
GP_Posterior_Mean 0.422624 0.350863 0.359345 0.422000               0.009737               0.009737  0.644108 0.095780  0.282232   0.229520    0.800466   0.086599
            GP_EI 0.449505 0.333625 0.367610 0.455359               0.053613               0.053613  0.296419 0.475891 -0.208457 